In [1]:
# The predictions of our previous model (in v4_NN) are not good because we take only one character of context

In [2]:
# The size of the count array table N grows exponentially with the length of the context
# If we take single character tp predict the next word like we were doing before, its 27 possibilities for context
# If we take 2 characters as context, it is 27*27 = 729 possibilities for context
# If we take 3 ->> 27*27*27 = 19,683 possibilities for context

# Multi Layer Perceptron Approach by Bengio et al. 2003

In [3]:
# They used vocab_size = 17k and embedding_dim = 30
# The embeddings will be trained during training of the Neural Network.

In [4]:
# You can transfer knowledge through the emdedding space, because once the embeddings are trained they contain similar words in similar part of the embedding space and many different corelations between words that were learned during the training phase of one language model can be used as a starting point for another laguage model.

In [5]:
# C is [17000, 30] #embedding vector


# Building an MLP: From a Bigram Model to a Neural Network

## 1. Objective

- We are taking **three previous words** and trying to predict the **fourth word** in a sequence.
- This is a step up from the bigram model, where we only considered the previous word to predict the next word.
- We now use a neural network to learn patterns from a larger context.

## 2. Input Representation: Word Indices

- We have a vocabulary of **17,000 possible words**.
- Each incoming word is represented by its index in the vocabulary.
- Since there are 17,000 words, each index is an integer between:

$$
0 \text{ and } 16,999
$$

- These indices are used to look up the corresponding word embeddings.

## 3. Embedding Matrix (C)

- There is a lookup table called **C**, which is a matrix of size:

$$
C \in \mathbb{R}^{17000 \times 30}
$$

- We treat this matrix as a lookup table.
- Each word index plucks out a row from the embedding matrix.
- Each row is a 30-dimensional vector corresponding to the embedding of that word.

In other words:

- Input: An integer representing a word's index.
- Output: A 30-dimensional embedding vector.

**Important:** The same embedding matrix C is shared across all three input words. We are always indexing into the same matrix C for each word.

## 4. Input Layer

- Each of the three previous words is converted into a 30-dimensional embedding vector.
- These three embedding vectors are concatenated together.

Therefore, the total number of input neurons is:

$$
3 \times 30 = 90
$$

So, the input layer consists of **90 neurons**.

## 5. Hidden Layer

- Next comes the hidden layer of the neural network.
- The size of this hidden layer is a **hyperparameter**.
- A hyperparameter is essentially a design choice made by the designer of the neural network.
- We can choose this size to be as large or as small as we like.

For example, suppose we choose:

$$
\text{Hidden layer size} = 100
$$

- All 100 neurons are fully connected to the 90 inputs.
- This is called a **fully connected layer**.

After this layer, we apply a `tanh` nonlinearity.

$$
h = \tanh(W_1 x + b_1)
$$

Where:
- $x$ is the 90-dimensional concatenated embedding vector.
- $W_1$ represents the weights connecting the input layer to the hidden layer.
- $b_1$ represents the biases.
- $h$ is the resulting hidden-layer activation.

## 6. Output Layer

- After the hidden layer comes the output layer.
- Since there are 17,000 possible words that could come next, the output layer has **17,000 neurons**.
- Every neuron in the output layer is fully connected to every neuron in the hidden layer.

The output layer produces 17,000 logits:

$$
\text{logits} = W_2 h + b_2
$$

Where:
- $W_2$ represents the weights connecting the hidden layer to the output layer.
- $b_2$ represents the biases.
- Each logit corresponds to one possible word in the vocabulary.

**Important:** This is the expensive layer because there are so many possible words, resulting in a large number of parameters and a significant amount of computation.

## 7. Softmax Layer

- We have 17,000 logits coming out of the output layer.
- We apply the softmax function to convert these logits into a probability distribution.
- Every logit is exponentiated, and then everything is normalized so that the probabilities sum to one.

$$
P(y_i) = \frac{e^{z_i}}{\sum_{j=1}^{17000} e^{z_j}}
$$

Where:
- $z_i$ is the logit corresponding to word $i$.
- $P(y_i)$ is the predicted probability of word $i$ being the next word.

The result is a probability distribution over all 17,000 possible next words.

## 8. Training: Using the Actual Next Word

- During training, we already know the actual next word in the sequence. This is our **label**.
- The identity (index) of that next word is used to pluck out its corresponding probability from the softmax output.
- We then maximize the probability assigned to the correct next word with respect to the parameters of the neural network.

Equivalently, we minimize the negative log probability of the correct word:

$$
L = -\log P(\text{correct next word})
$$

The network learns by adjusting its parameters to increase the probability of the correct next word.

---

## Overall Architecture

The complete flow looks like this:

1. **Three previous words:** Represented by integer indices from 0 to 16,999.
2. **Embedding lookup (C):** Each index retrieves a 30-dimensional embedding vector.
3. **Concatenation:** Three embeddings are concatenated into a 90-dimensional input.
4. **Hidden layer:** A fully connected layer with 100 neurons (example hyperparameter).
5. **Nonlinearity:** Apply `tanh`.
6. **Output layer:** A fully connected layer with 17,000 neurons producing logits.
7. **Softmax:** Convert logits into a probability distribution over the vocabulary.
8. **Training:** Use the actual next word's index to retrieve its probability and maximize that probability.

**Note:** The hidden layer size of 100 and embedding dimension of 30 are example design choices, not fixed requirements.


In [8]:
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt

In [9]:
words = open('/content/sample_data/names.txt').read().splitlines()

In [10]:
words[:8]

['emma', 'olivia', 'ava', 'isabella', 'sophia', 'charlotte', 'mia', 'amelia']

In [11]:
len(words)

32033

In [29]:
chars = sorted(list(set(''.join(words))))
stoi = {s:i+1 for i, s in enumerate(chars)}
stoi['.'] = 0
itos = {i:s for s, i in stoi.items()}
print(stoi)
print(itos)

{'a': 1, 'b': 2, 'c': 3, 'd': 4, 'e': 5, 'f': 6, 'g': 7, 'h': 8, 'i': 9, 'j': 10, 'k': 11, 'l': 12, 'm': 13, 'n': 14, 'o': 15, 'p': 16, 'q': 17, 'r': 18, 's': 19, 't': 20, 'u': 21, 'v': 22, 'w': 23, 'x': 24, 'y': 25, 'z': 26, '.': 0}
{1: 'a', 2: 'b', 3: 'c', 4: 'd', 5: 'e', 6: 'f', 7: 'g', 8: 'h', 9: 'i', 10: 'j', 11: 'k', 12: 'l', 13: 'm', 14: 'n', 15: 'o', 16: 'p', 17: 'q', 18: 'r', 19: 's', 20: 't', 21: 'u', 22: 'v', 23: 'w', 24: 'x', 25: 'y', 26: 'z', 0: '.'}


In [30]:
stoi['e']

5

In [37]:
# build the dataset

block_size = 3 # context length: how many characters do we take to predit the next character.
X, Y = [], []

for wd in words[:1]:

    print(wd)
    context = [0]* block_size
    new_wd = wd + '.'
    for ch in new_wd:
        ix = stoi[ch]
        print(f"ch = {ch}, ix = {ix} ")
        print(f"context = {context}   y_true = {ix}")
        X.append(context)
        Y.append(ix)
        context = context[1:] + [ix] # Crop and append, this is the main thing being done here. we shift exisitng characters left by 1 and add the newest cahracter to the right. The left most character gets cut off

emma
ch = e, ix = 5 
context = [0, 0, 0]   y_true = 5
ch = m, ix = 13 
context = [0, 0, 5]   y_true = 13
ch = m, ix = 13 
context = [0, 5, 13]   y_true = 13
ch = a, ix = 1 
context = [5, 13, 13]   y_true = 1
ch = ., ix = 0 
context = [13, 13, 1]   y_true = 0


**Note:** we are not doing ```.emma.```  and then doing indexing. But instead we are just doing ```emma.```. This is beacuase the `.` at the beginning is being handled by the padded context of `[0,0,0]` that we start with.

The thing gets handled automatically without us having to set is manually

In [42]:
# build the dataset - For real this time

block_size = 3 # context length: how many characters do we take to predit the next character.
X, Y = [], []

for w in words[:5]:

    print(w)
    context = [0]* block_size

    for ch in w + '.':
        ix = stoi[ch]
        X.append(context)
        Y.append(ix)
        print(''.join(itos[i] for i in context), '--->', itos[ix])
        context = context[1:] + [ix] # Crop and append, this is the main thing being done here. we shift exisitng characters left by 1 and add the newest cahracter to the right. The left most character gets cut off

X = torch.tensor(X)
Y = torch.tensor(Y)

emma
... ---> e
..e ---> m
.em ---> m
emm ---> a
mma ---> .
olivia
... ---> o
..o ---> l
.ol ---> i
oli ---> v
liv ---> i
ivi ---> a
via ---> .
ava
... ---> a
..a ---> v
.av ---> a
ava ---> .
isabella
... ---> i
..i ---> s
.is ---> a
isa ---> b
sab ---> e
abe ---> l
bel ---> l
ell ---> a
lla ---> .
sophia
... ---> s
..s ---> o
.so ---> p
sop ---> h
oph ---> i
phi ---> a
hia ---> .


In [43]:
# # build the dataset - No, but For real, for real this time

# block_size = 3 # context length: how many characters do we take to predit the next character.
# X, Y = [], []

# for w in words:

#     #print(w)
#     context = [0]* block_size

#     for ch in w + '.':
#         ix = stoi[ch]
#         X.append(context)
#         Y.append(ix)
#         # print(''.join(itos[i] for i in context), '--->', itos[ix])
#         context = context[1:] + [ix] # Crop and append, this is the main thing being done here. we shift exisitng characters left by 1 and add the newest cahracter to the right. The left most character gets cut off

# X = torch.tensor(X)
# Y = torch.tensor(Y)

In [44]:
X.shape, X.dtype, Y.shape, Y.dtype

(torch.Size([32, 3]), torch.int64, torch.Size([32]), torch.int64)

In [ ]:
# We'll use embedding_dim = 2

In [45]:
C = torch.randn((27, 2))

In [46]:
C[5]

tensor([ 0.6868, -0.5165])

In [47]:
F.one_hot(torch.tensor(5), num_classes=27).float() @ C

tensor([ 0.6868, -0.5165])

In [48]:
# indexing from embedding_matrix C is equivalent to onehot matrix multiplication.
# We'll use indexing as its faster.

In [49]:
# Pytorch indexing is very powerful, we can index with a list, or even a tensor of integers

In [50]:
# We can also so 2D tesnor indexing

In [51]:
C[X].shape

torch.Size([32, 3, 2])

In [53]:
emb = C[X]
emb.shape

torch.Size([32, 3, 2])

In [54]:
# Let's initalize the Weights of Layer 1 (i,e,layer_10 - from 0 to 1)

# For our example where we have context length = 3 (same as paper), but embedding_dim = 2, the total number of inputs in a[0], ie x is 6.

# So layer 0 has 6 neurons
# Layer 1 has 100 neurons

In [55]:
W1 = torch.randn(6,100)
b1 = torch.randn(100)

In [ ]:
# We need to do emb @ W1 + b1
# But emb = [32,3,2] and W1 = [6,100]
# So we need to reshape using view function.
# emb = emb.view(32, 6)
# Then ->>>   emb @ W1 + b1

In [58]:
# One bad way to do reshape using cat
torch.cat((emb[:,0,:], emb[:,0,:], emb[:,0,:]), dim = 1).shape

torch.Size([32, 6])

In [64]:
# Another bad way
unbinded = torch.unbind(emb, 1)
print(type(unbinded))
print(len(unbinded))
print(unbinded[0].shape)

<class 'tuple'>
3
torch.Size([32, 2])


In [ ]:
# This (torch.unbind(emb, 1)) converts [32, 3, 2] to 3 tensors of size [32, 2].
# The middle dimension i.e, dim=1 is now the number of tupes we get back.

In [71]:
torch.cat(torch.unbind(emb, 1), dim =1).shape # this is infficeint because this creates new memory

torch.Size([32, 6])

In [67]:
 # .view() is extemently efficient in PyTroch

In [68]:
h = emb.view(32,6) @ W1 + b1 # 32 is being hard coded, remove this

In [69]:
h.shape

torch.Size([32, 100])

In [72]:
h = emb.view(-1, 6) @ W1 + b1
# When we do -1, Pytoch will infer what this must be
# Then apply the activation function

In [73]:
h = torch.tanh(emb.view(32,6) @ W1 + b1)

In [74]:
print((emb.view(32,6) @ W1).shape)
print(b1.shape)

torch.Size([32, 100])
torch.Size([100])


In [75]:
# 32, 100
#  1, 100

In [79]:
W2 = torch.randn((100,27))
b2 = torch.randn(27)

In [80]:
logits = h @ W2 + b2

In [81]:
logits.shape

torch.Size([32, 27])

In [82]:
counts = logits.exp()
prob = counts/ counts.sum(dim=1, keepdims=True)

In [83]:
prob.shape

torch.Size([32, 27])

In [84]:
Y

tensor([ 5, 13, 13,  1,  0, 15, 12,  9, 22,  9,  1,  0,  1, 22,  1,  0,  9, 19,
         1,  2,  5, 12, 12,  1,  0, 19, 15, 16,  8,  9,  1,  0])

In [85]:
prob[torch.arange(32), Y]

tensor([1.0564e-09, 1.3247e-09, 3.3439e-03, 4.8119e-04, 8.6497e-07, 4.8765e-06,
        2.2003e-14, 5.4700e-08, 3.7292e-13, 1.2638e-05, 2.7185e-12, 9.7121e-01,
        7.6684e-10, 2.2464e-09, 7.2561e-04, 5.9307e-06, 9.9711e-01, 5.0746e-16,
        4.4369e-07, 1.2206e-01, 5.8490e-10, 1.0937e-09, 4.7450e-01, 4.6276e-10,
        4.1374e-14, 3.0977e-14, 1.9264e-06, 6.3448e-02, 2.0603e-03, 7.3752e-04,
        1.5884e-12, 9.8214e-01])

In [86]:
nll = -prob[torch.arange(32), Y].log().mean()
nll

tensor(15.3138)

In [87]:
# ----------- now more respactable :)------------

In [88]:
X.shape, Y.shape  #dataset

(torch.Size([32, 3]), torch.Size([32]))

In [89]:
g = torch.Generator().manual_seed(2147483647) # for reproducibility
C = torch.randn((27, 2), generator=g)
W1 = torch.randn((6, 100), generator=g)
b1 = torch.randn(100, generator=g)
W2 = torch.randn((100, 27), generator=g)
b2 = torch.randn(27, generator=g)
parameters = [C, W1, b1, W2, b2]

In [91]:
sum(p.nelement() for p in parameters) #number of parameters in total

3481

In [95]:
emb = C[X] # [32, 3, 2]
h = torch.tanh(emb.view(32,6) @ W1 + b1)    # [32, 100]
logits = h @ W2 + b2                        #[32, 27]
counts = logits.exp()
prob = counts/ counts.sum(dim=1, keepdims=True)
loss = -prob[torch.arange(32), Y].log().mean()
loss

tensor(17.7697)

These 3 lines
```python
counts = logits.exp()
prob = counts/ counts.sum(dim=1, keepdims=True)
loss = -prob[torch.arange(32), Y].log().mean()
```

can be written using

```
F.cross_entropy(logits,Y)
```

In [96]:
emb = C[X] # [32, 3, 2]
h = torch.tanh(emb.view(32,6) @ W1 + b1)    # [32, 100]
logits = h @ W2 + b2                        #[32, 27]
F.cross_entropy(logits,Y)
loss

tensor(17.7697)


# Cross Entropy Loss in PyTorch

## 1. Using PyTorch's Built-in Cross Entropy Function

- We are not reinventing the wheel when calculating the loss. This is simply a **classification problem**.
- Classification is a very common task, which is why PyTorch provides a built-in function:

```python
F.cross_entropy(logits, y)
```

- We pass two arguments:
  - `logits`: The raw output of the neural network.
  - `y`: The array of target labels (the actual next words).
- This calculates the exact same loss as our manually implemented cross-entropy calculation.

Therefore, we can replace our manual implementation with `F.cross_entropy()`.

## 2. Why Prefer `F.cross_entropy()` Over a Custom Implementation?

There are several good reasons to prefer PyTorch's built-in implementation over manually implementing the loss.

### Reason 1: Memory and Computational Efficiency

- When we manually implement cross-entropy, PyTorch creates several intermediate tensors in memory.
- Each operation produces new tensors, making the implementation relatively inefficient.
- Instead, PyTorch can cluster these operations together and often use **fused kernels** to evaluate them efficiently.
- A fused kernel combines multiple mathematical operations into a single, optimized operation.

This reduces unnecessary intermediate tensor creation and improves computational efficiency.

### Reason 2: More Efficient Backward Pass

- The backward pass can also be made much more efficient.
- This is not just because of fused kernels. The mathematical expression for the derivative can often be simplified analytically.

**Example from Micrograd:**

When we implemented `tanh`, its forward pass involved a relatively complicated mathematical expression.

However, during the backward pass, we didn't individually backpropagate through every operation:

- Exponentiation (`exp`)
- Multiplication
- Subtraction
- Division

Instead, we used the simplified derivative:

$$
\frac{d}{dx}\tanh(x) = 1 - \tanh^2(x)
$$

This is a much simpler mathematical expression.

We can do this because:
- We can reuse calculations from the forward pass.
- We can analytically derive the derivative.
- The resulting mathematical expression often simplifies significantly.

**Key takeaway:** Built-in PyTorch operations can be more efficient not only because they use fused kernels, but also because their backward passes can use mathematically simplified derivatives instead of individually differentiating through every underlying operation.



## 3. Numerical Stability: Why Built-in Cross Entropy Is Safer

Another important reason to prefer PyTorch's `F.cross_entropy()` is that it is significantly more **numerically stable**.

### The Problem with Manual Softmax and Cross Entropy

Consider our manual implementation:

```python
counts = logits.exp()
prob = counts / counts.sum(1, keepdims=True)
loss = -prob[torch.arange(32), Y].log().mean()
```

Here, we:
1. Exponentiate the logits.
2. Normalize them to obtain probabilities.
3. Extract the probability of the correct target.
4. Take the negative logarithm and average the loss.

This works well when logits have reasonable values.

For example, consider:

$$
\text{logits} = [-2, -3, 0, 5]
$$

Exponentiating and normalizing these values produces a valid probability distribution.

### What Happens When Logits Become Extreme?

During neural network optimization, logits can sometimes take on extreme values.

Consider two cases:

**Case 1: Very negative logits**

Suppose one of the logits becomes $-100$.

$$
e^{-100} \approx 0
$$

This is not necessarily a problem. The exponential produces a very small number close to zero, and the resulting probability distribution can still be well behaved.

**Case 2: Very positive logits**

Now suppose one of the logits becomes $100$.

$$
e^{100}
$$

This can exceed the representable range of floating-point numbers.

As a result:

$$
e^{100} = \infty
$$

We encounter an overflow problem.

When we subsequently normalize the counts, we can end up with:

$$
\frac{\infty}{\infty} = \text{NaN}
$$

So our manually implemented softmax and cross-entropy calculation can break when logits become too large.

### The Solution: Subtract the Maximum Logit

There is an interesting mathematical property of softmax.

**We can add or subtract any arbitrary constant from all logits without changing the resulting probabilities.**

To understand why, consider softmax:

$$
P_i = \frac{e^{z_i}}{\sum_j e^{z_j}}
$$

Suppose we subtract a constant $c$ from every logit:

$$
P_i = \frac{e^{z_i-c}}{\sum_j e^{z_j-c}}
$$

Using the exponential property:

$$
P_i = \frac{e^{z_i}e^{-c}}{e^{-c}\sum_j e^{z_j}}
$$

The common factor $e^{-c}$ cancels out:

$$
P_i = \frac{e^{z_i}}{\sum_j e^{z_j}}
$$

Therefore, the probabilities remain exactly the same mathematically.

### How PyTorch Uses This Property

PyTorch internally calculates the maximum logit and subtracts it from every logit before performing the exponential operation.

For example:

$$
\text{Original logits} = [-2, -3, 0, 5]
$$

Maximum logit:

$$
\max(\text{logits}) = 5
$$

Subtracting 5 from every logit:

$$
\text{Adjusted logits} = [-7, -8, -5, 0]
$$

Now the largest logit is always zero, and all the remaining logits are negative.

Consequently:
- The largest exponential value is $e^0 = 1$.
- All other exponential values are between 0 and 1.
- We avoid exponential overflow.

Even if the original logits contained a value of 100, subtracting the maximum (100) would bring the largest adjusted logit back to zero.

This technique is called **numerically stable softmax computation** and is commonly known as the log-sum-exp trick.

## 4. Summary: Why Use `F.cross_entropy()`?

There are three major reasons to prefer PyTorch's built-in cross-entropy function over manually implementing softmax and cross entropy.

1. **Efficient forward pass:** PyTorch can combine operations using fused kernels, reducing intermediate tensor creation and improving computational efficiency.

2. **Efficient backward pass:** The derivative can be analytically simplified, resulting in a more efficient backward computation.

3. **Numerical stability:** PyTorch uses techniques such as subtracting the maximum logit before exponentiation to prevent numerical overflow.

**Important:** Although implementing softmax and cross entropy manually is useful for understanding the mathematics, we should generally use `F.cross_entropy()` in practice.

```python
import torch.nn.functional as F

loss = F.cross_entropy(logits, Y)
```

PyTorch handles the softmax and cross-entropy calculation internally. We pass the raw logits, not the softmax probabilities.

---

## 5. Next Step: Training the Neural Network

Now that we have established the loss calculation using `F.cross_entropy()`, we can set up the training process for our neural network.


In [100]:
X.shape, Y.shape  #dataset

(torch.Size([32, 3]), torch.Size([32]))

In [101]:
g = torch.Generator().manual_seed(2147483647) # for reproducibility
C = torch.randn((27, 2), generator=g)
W1 = torch.randn((6, 100), generator=g)
b1 = torch.randn(100, generator=g)
W2 = torch.randn((100, 27), generator=g)
b2 = torch.randn(27, generator=g)
parameters = [C, W1, b1, W2, b2]

In [103]:
for p in parameters:
  p.requires_grad = True

In [106]:
for _ in range(1000):

    # forward pass
    emb = C[X] # [32, 3, 2]
    h = torch.tanh(emb.view(32,6) @ W1 + b1)    # [32, 100]
    logits = h @ W2 + b2                        #[32, 27]
    loss = F.cross_entropy(logits,Y)

    #backward pass
    for p in parameters:
        p.grad = None
    loss.backward()

    # update
    for p in parameters:
        p.data += -0.1 * p.grad

print(loss.item())

0.2561509907245636



## 6. Training the Neural Network: Overfitting a Single Batch

- We now have the loss calculation in place and can start training the neural network.
- Initially, we start with a loss of approximately **17**.
- As we run the training iterations, the loss keeps decreasing.
- If we run the training for around 1,000 iterations, we get a very, very low loss.

This means that the network is making very good predictions on the training examples.

### Why Does the Loss Decrease So Quickly?

The reason this is so straightforward right now is that we are **overfitting just 32 examples**.

- We are only considering 32 examples (the first five words).
- Our neural network has approximately 3,400 parameters.
- We have significantly more parameters than training examples.

Therefore, it is very easy for the neural network to fit these 32 examples.

This is called **overfitting a single batch of data**.

We are essentially training the network to memorize these 32 examples, resulting in a very low loss and good predictions on those same examples.

**Important:** A very low training loss does not necessarily mean that our model has learned to generalize well. It may simply have memorized the training examples.

## 7. Why Can't We Achieve Exactly Zero Loss?

Even after running many iterations, we are not able to achieve exactly zero loss.

To understand why, Karpathy starts inspecting the logits being predicted by the neural network.

### Inspecting the Predicted Logits

- We can look at the maximum logit along the first dimension.
- In PyTorch, `max()` reports both:
  - The actual maximum values.
  - The indices at which those maximum values occur.

For example:

```python
logits.max(1)
```

This helps us inspect which output neurons (corresponding to vocabulary words) are producing the highest logits.

We can compare the predicted indices with the actual target labels.

- In most cases, the predicted indices are very close to the labels.
- However, in some cases, they differ.

For example, in the very first example:
- Predicted index = 19
- Actual label = 5

### The Fundamental Reason: Ambiguous Input-Output Mappings

The reason we cannot make the loss exactly zero is that **the same input context can have multiple possible next words in the training dataset**.

For example, suppose our input context is:

$$
(\text{...}, \text{...}, \text{...})
$$

In different examples from the dataset, this exact same input context is followed by different words:

- In one example, the next word is E.
- In another example, the next word is O.
- In another example, the next word is A.
- In another example, the next word is S.

Therefore, the same input is associated with multiple possible outputs.

The neural network cannot assign a probability of 1 to all these different outcomes simultaneously.

If it predicts E with probability 1, it cannot also predict O, A, and S with probability 1 for the exact same input.

Consequently, the loss cannot reach exactly zero.

### When Can We Overfit Completely?

- In cases where a unique input corresponds to a unique output, we can overfit the example.
- In those cases, the network can learn to produce the exact correct result.
- However, when the same input corresponds to multiple possible outputs, we cannot completely overfit all the examples simultaneously.

**Key takeaway:** The inability to achieve exactly zero loss is not necessarily a limitation of the neural network. It can be a consequence of ambiguity in the training data itself.

## 8. Next Step: Training on the Full Dataset

So far, we have only used the first five words to construct our dataset and have been overfitting a single batch of 32 examples.

The next step is to:

1. Go back to where we created the dataset.
2. Read in the full dataset instead of using only the first five words.
3. Optimize the neural network using the complete dataset.

This will allow us to train the network on many more examples rather than simply memorizing a small batch.
